# 🎯 [OPS04_M10] 샤헤드·미사일 포화경보 우선순위 시스템 최종 통합 및 방공작전 브리핑 패키지 완성
### Google Colab 실습 환경 (작전·전투 수행 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/OPS04/ops04_m10_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 작전·전투 수행 (Level 3) |
| **미션 ID** | OPS04_M10 |
| **미션 주제** | 샤헤드·미사일 포화경보 우선순위 시스템 최종 통합 및 방공작전 브리핑 패키지 완성 |
| **소요 시간** | 약 3.5시간 |
| **핵심 산출물** | `src/main_ops04.py, README.md, docs/ops04_final_report.pdf(md)` |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

## 📂 2. 표준 산출물 디렉토리 체계 및 실습 팁
- **방법 A (`%%writefile` 저장 추천):** 코드 셀 첫 줄에 `%%writefile <산출물 경로>`(예: `%%writefile src/models/ops04_baseline_model.py`)를 추가해 해당 폴더에 모듈을 저장할 수 있습니다.
- **방법 B (셀 단위 논리 대응 인정):** 별도 파일 추출 없이 노트북 셀 단위로 독립 실행·검증하고 미션 10 통합을 완료해도 정상 인정됩니다.

> **⚠️ [교육용 가상 데이터셋 및 HITL 원칙]**
> - 본 노트북의 모든 좌표·비행체 유형·수치는 보안 규정을 준수하여 **교육용 가상 데이터로 생성**됩니다.
> - 실제 무기 제원·사거리·부대 위치·군사 기밀 데이터는 일체 사용하지 않습니다.
> - 산출되는 대응 순위는 지휘관 승인을 위한 **'권고'**이며 자동 승인·자동 실행은 없습니다.

---

In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 10. 샤헤드·미사일 포화경보 우선순위 시스템 최종 통합 및 방공작전 브리핑 패키지 완성
**소요시간**: 약 3.5시간 | **이번 산출물**: `src/main_ops04.py, README.md, docs/ops04_final_report.pdf(md)`

### 📋 미션 개요
**미션 1~9에서 구축한 검증·전처리·다차원 평가·신뢰도·대시보드·HITL·재보정 모듈을 단일 파이프라인으로 통합 실행하고, 자체 점검과 지휘관 결심 지원용 브리핑 패키지를 완성합니다.**

- **배경 및 필요성:** 개별 모듈이 각각 통과하더라도 End-to-End 연동 과정에서 스키마 불일치나 재보정 파라미터 미반영 문제가 발생할 수 있으므로, 전 구간 통합 실행과 7대 자체 점검을 통해 실전형 프로토타입 완성도를 최종 입증해야 합니다.
- **핵심 과업:** 재보정 파라미터가 반영된 통합 파이프라인을 한 파일로 묶어 가상 상황 5종을 일괄 처리하고, 자체 점검 7개 항목(손상 격리, 스푸핑 경보 0건, 자동 승인 0건, 재현율, 처리시간, 커버리지 등)을 판정한 뒤 브리핑 요약 5개 항목을 작성합니다.
- **학습 목표 (습득 역량):** End-to-End AI 전술 의사결정 파이프라인 통합 아키텍처 구성 능력, 통합 회귀·커버리지 자체 점검 역량, 지휘관 결심 지원을 위한 정량 근거 기반 작전 브리핑 작성 능력을 체득합니다.
- **검증 및 산출물:** 통합 파이프라인 실행 성공률 100%, 테스트 커버리지 85% 이상을 만족하는 통합 코드(`src/main_ops04.py`), `README.md`, 최종 보고서(`docs/ops04_final_report.pdf(md)`)를 산출합니다.

### 🎓 학습 목표 (습득 역량)
**[습득 역량]** 개별 미션에서 구현한 데이터 검증·전처리·재보정 위협 평가·신뢰도·권고 순위·HITL 승인 모듈을 단일 End-to-End 파이프라인으로 통합하고, 정량 자체 점검과 지휘관 결심 지원용 작전 브리핑을 완성하는 종합 시스템 역량을 체득합니다.

**[핵심 과업]** 전체 파이프라인을 한 파일로 통합해 가상 상황 5종을 처리하고 손상 격리·스푸핑 경보 0건·자동 승인 0건·재현율·처리시간·함수 커버리지를 자체 점검한 뒤, 실행 수치와 운용 한계를 포함한 상황판·브리핑 요약을 완성합니다.

- **End-to-End 전술 파이프라인 통합 아키텍처 역량:** 앞 단계의 스키마·전처리·재보정 가중치·신뢰도·HITL 상태가 데이터 규격 불일치 없이 단일 실행 흐름으로 유기적으로 연동되도록 결합하는 시스템 통합 능력을 익힙니다.
- **다중 시나리오 회귀 및 커버리지 자체 점검 능력:** 가상 상황 5종에 대해 7개 자체 점검 항목(실행 성공률 100%, 자동 승인 0건, 함수 커버리지 85% 이상 등)을 자동 검증해 프로토타입의 신뢰성을 입증하는 QA 기술을 습득합니다.
- **정량 근거 기반 지휘관 브리핑 패키지 작성 역량:** 산출된 성능 수치와 함께 '권고는 지휘관 승인 대상'이라는 HITL 원칙 및 추상 가상 데이터 기반 모델의 한계를 명확히 보고하는 실무 의사소통 역량을 완성합니다.

### 📖 미션 배경 및 시나리오
#### 미션 배경 및 당위성
샤헤드·미사일 포화경보 우선순위 시스템을 최종 통합하고 지휘관 결심 지원용 실전 브리핑 패키지와 운영 매뉴얼을 완성한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **전체 파이프라인을 통합해 실행·자체 점검하고 지휘관 결심 지원용 브리핑 패키지를 완성한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **검증·전처리, 다차원 평가(재보정값), 신뢰도 등급, 권고 순위, 승인 대기열을 한 파일로 묶어 가상 상황 5개를 처리 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/main_ops04.py, README.md, docs/ops04_final_report.pdf(md)` 형태의 결과물을 완성한다.

### 🤖 단계별 수행 및 AI 협업 가이드
#### Step 1. 미션 10 코드를 실행해 5개 상황 통합 결과를 확인한다.
#### 이 단계의 역할
미션 10의 통합 `run()` 파이프라인을 실행해 검증·전처리·재보정 다차원 평가·신뢰도·권고 순위·승인 대기열이 가상 상황 5종을 무오류로 처리하는지 확인합니다.

#### 해야 할 일

- 재보정된 파라미터(`CONFIG`)가 적용된 단일 통합 실행 흐름(`validate -> preprocess -> assess -> recommend -> run`)을 점검한다.
- AI를 Co-pilot으로 활용해 가상 상황 5종(평균 처리 31.0 ms, 상황 1 기준 정확도 94.5%, 긴급 재현율 100.0%)의 입출력 스키마 연결 정합성을 확인한다.
- 상황 5개 중 성공 5개(성공률 100%)와 손상 격리·중복 제거 통계를 확인한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "validate, preprocess, assess(재보정 파라미터 적용), recommend, HITL 승인 대기열을 하나로 묶은 run() 통합 파이프라인이 가상 상황 5종을 오류 없이 통과하는지 단계별 입출력 타입과 처리시간을 점검하는 진단 코드를 작성해줘."

  - **AI 결과 검증 포인트:** 앞 미션의 출력 키 이름과 뒤 미션의 입력 키 이름이 어긋나는 구간이 없는지, 그리고 재보정된 `CONFIG` 값이 실제로 `assess()`에 반영되었는지 확인합니다.

#### 핵심 확인 지표

> 상황 5/5 성공 (실행 성공률 100%), 평균 처리 31.0 ms, 판정 정확도 94.5%, 긴급 표적 경보 재현율 100.0%

#### 학습자가 제출해야 하는 내용

- 미션 10 통합 실행 결과 및 가상 상황 5종 처리 로그
- AI 프롬프트 질의 및 모듈 간 인터페이스 정합성 검증 기록

#### Step 2. 자체 점검 7개 항목의 PASS/FAIL을 확인한다.
#### 이 단계의 역할
자체 점검 7개 항목(실행 성공률·손상 격리·스푸핑 경보 0건·자동 승인 0건·재현율 90% 이상·예산 준수·함수 커버리지 85% 이상)의 PASS/FAIL을 검증합니다.

#### 해야 할 일

- 통합 코드 말미의 7개 자체 점검 체크리스트 실행 결과와 함수 커버리지(100%, 목표 85% 이상)를 확인한다.
- AI에게 7개 점검 항목 각각의 PASS 조건과 실패 시 디버깅 포인트를 매핑하는 자동 검증 요약표 작성을 요청한다.
- 7/7 PASS 달성과 모든 권고 항목의 '승인 대기'(자동 승인 0건) 상태를 확인한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "통합 파이프라인의 자체 점검 7개 항목(E2E 성공률 100%, 손상 전량 격리, 스푸핑 경보 0건, 자동 승인 0건, 긴급 재현율 90% 이상, 처리시간 예산 이내, 함수 커버리지 85% 이상)의 검증 로직이 형식적 통과가 아닌 실질적 검증인지 코드 리뷰해줘."

  - **AI 결과 검증 포인트:** AI의 코드 리뷰를 통해 자체 점검 조건이 하드코딩된 True가 아니라 실제 실행 결과값(`assert`)을 검사하는지, 함수 커버리지가 미사용 함수를 제외하지 않았는지 확인합니다.

#### 핵심 확인 지표

> 자체 점검 7/7 PASS, 스푸핑 경보 0건, 자동 승인 0건, 함수 커버리지 100% (목표 85% 이상)

#### 학습자가 제출해야 하는 내용

- 자체 점검 7개 항목 PASS/FAIL 실행 로그 및 커버리지 리포트
- AI 코드 리뷰 기반 검증 신뢰성 점검 기록

#### Step 3. 브리핑 요약 5개 항목을 실행 결과 수치와 대조한다.
#### 이 단계의 역할
포화경보 우선순위 브리핑 요약 5개 항목(상황·성능·권고·운용 유의·한계)을 실제 실행 수치와 1:1 대조해 최종 브리핑 패키지를 완성합니다.

#### 해야 할 일

- 출력된 브리핑 요약 5개 항목의 수치(평가 124건, 긴급 30/주의 30/일반 64, 정확도 94.5%, 재현율 100.0%, 권고 60건 승인 대기)를 실행 로그와 대조한다.
- AI에게 `README.md` 실행 가이드와 `docs/ops04_final_report.md` 최종 브리핑 패키지의 목차·보안 한계 고지 문구 다듬기를 요청한다.
- 최종 산출물(`src/main_ops04.py`, `README.md`, `docs/ops04_final_report.pdf(md)`)에 추상 모델 한계와 지휘관 승인 원칙이 명시되었는지 확인한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "미션 10 실행 출력 수치(평가 124건, 정확도 94.5%, 긴급 재현율 100.0%, 권고 60건 승인 대기, 자동 승인 0건)를 바탕으로 지휘관 브리핑 패키지(ops04_final_report.md)와 실행 README.md 요약을 작성해줘. 권고는 지휘관 승인 대상이며 가상 데이터 기반 한계를 반드시 명시해줘."

  - **AI 결과 검증 포인트:** AI가 작성한 브리핑 문서의 모든 수치가 실제 실행 결과(예: 처리 29.1 ms, 정확도 94.5%)와 일치하는지 교차 대조하고, 과장된 실전 도입 표현을 배제했는지 확인합니다.

#### 핵심 확인 지표

> 브리핑 요약 5개 항목 수치 100% 일치, 최종 브리핑 패키지(README·최종 보고서) 완성

#### 학습자가 제출해야 하는 내용

- 브리핑 요약 5개 항목 대조표 및 최종 보고서(`docs/ops04_final_report.md`)
- AI 작성 문서 수치 팩트체크 및 HITL 원칙 반영 기록

#### Step 4. 도전 과제 및 파라미터 튜닝 최적화
#### 이 단계의 역할
`CONFIG`의 `w_spoof` 가중치를 `0.0`으로 설정했을 때 '스푸핑 의심 항적 경보 0건' 자체 점검이 어떻게 실패하는지 확인해 안전 가중치의 필수성을 입증합니다.

#### 해야 할 일

- `CONFIG`의 `w_spoof` 값을 `0.0`으로 변경해 통합 `run()`을 재실행한다.
- AI에게 스푸핑 감점이 0일 때 스푸핑 항적이 경보로 유입되어 자체 점검 7개 항목 중 어느 항목이 FAIL로 전환되는지 원인 분석을 요청한다.
- `w_spoof` 원상 복구 후 7/7 PASS 회복 결과와 파라미터 민감도 분석 메모를 작성한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "통합 CONFIG에서 w_spoof를 0.0으로 바꿔 실행했을 때 '스푸핑 의심 항적 경보 0건' 자체 점검이 FAIL로 바뀌는 원인과 오경보 유입 건수를 분석하고, 복구 후 검증 결과를 비교 요약해줘."

  - **AI 결과 검증 포인트:** 의도적으로 안전 파라미터를 껐을 때 자체 점검이 실제로 FAIL을 탐지하는지(음성 대조군 검증) 확인해, 자체 점검 스위트의 결함 검출 신뢰성을 입증합니다.

#### 핵심 확인 지표

> w_spoof=0.0 시 '스푸핑 경보 0건' FAIL 검출 확인 및 복구 후 7/7 PASS 회복

#### 학습자가 제출해야 하는 내용

- w_spoof 변경 전후 자체 점검 비교 메모 및 최종 복구 검증 로그
- AI 질의 프롬프트 및 음성 대조군(FAIL 검출) 검증 기록

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정
**평가 구분:** 필수

**핵심 평가영역:** 군 실무 즉시 도입 가능성, 운용 신뢰도 및 전략적 기대효과

#### 미션 PASS 조건
- 정량 기준: End-to-End 통합 파이프라인 실행 성공률 100%, 테스트 커버리지 85% 이상
- 진행 기준: 원클릭 실행 스크립트 동작 확인 및 최종 발표 산출물 작성 완료
- 검증 방법: 최종 검증 위원회(평가관) 모의 시연 및 전체 기능 통합 검증
- AI 검증 기준: 통합 파이프라인 인터페이스 정합성, 7대 자체 점검 실질 검증(음성 대조군 포함) 및 브리핑 수치 팩트체크를 AI 협업으로 검증 완료
#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 AI 프롬프트 설계의 구체성, AI 산출물의 오류·보안 위반을 인간 검토(HITL)로 식별·수정한 기록, 그리고 출력 수치의 군 실무 부합성을 종합 채점한다.

In [ ]:
"""메시지 검증·전처리, 다차원 위협 평가(재보정 가중치), 신뢰도 등급, 권고 대응 순위, 지휘관 승인 대기열, 대시보드까지 전체 파이프라인을 한 파일로 통합 실행하고, 자체 점검(함수 커버리지 포함) 결과와 방공작전 브리핑 요약을 출력한다."""
import hashlib
import json
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ---------------- [교육용가상자료] 가상 방어지역·가상 비행체 생성기 ----------------
# 좌표는 가상 지휘소를 원점으로 한 km 단위 평면이다. 비행체 유형은 추상 범주이며
# 속도·고도·반사강도는 학습용 임의 범위로, 실제 무기 제원·사거리·탄도와 무관하다.
ASSETS = pd.DataFrame([["A1", "CP(virtual)", 0.0, 0.0, 1.0],
                       ["A2", "Radar(virtual)", 4.0, -3.0, 0.8],
                       ["A3", "Depot(virtual)", -5.0, 4.0, 0.7],
                       ["A4", "Power(virtual)", 7.0, 6.0, 0.5]], columns=["asset", "name", "x", "y", "value"])
TYPES = {  # 유형: (위협 여부, 속도 m/s 범위, 고도 m 범위, 출현 비율)
    "uav_type": (True, (30, 60), (100, 800), 0.42),      # 저속 무인기형(가상)
    "fast_type": (True, (150, 250), (50, 200), 0.16),    # 고속 순항형(가상)
    "decoy": (False, (30, 60), (100, 800), 0.16),        # 기만체(가상)
    "civil": (False, (120, 220), (3000, 9000), 0.10),    # 민항기(가상)
    "bird": (False, (8, 18), (50, 400), 0.16),           # 조류 군집
}
TTI_URGENT = 600      # 긴급 판정 기준: 가상 자산 도달 예상시간 600초 이내
CPA_KM = 1.5          # 자산 상공 통과 판정 반경(가상)


def make_raid(seed, n=80, spoof=0):
    """가상 포화 상황 1회분(비행체 n개 + 스푸핑 가짜 항적 spoof개)을 만든다."""
    rng = np.random.default_rng(seed)
    names = list(TYPES)
    rows = []
    for i in range(n + spoof):
        typ = "spoof" if i >= n else str(rng.choice(names, p=[TYPES[k][3] for k in names]))
        threat, sp, al, _ = TYPES["decoy" if typ == "spoof" else typ]
        brg = np.radians(rng.uniform(0, 360)); dist = rng.uniform(6, 60)
        x, y = dist * np.sin(brg), dist * np.cos(brg)
        spd = rng.uniform(*sp); alt = rng.uniform(*al)
        aim = int(rng.integers(0, len(ASSETS)))
        if typ in ("uav_type", "fast_type", "decoy", "spoof"):      # 가상 자산 방향(작은 방위 오차)
            ax_, ay_ = ASSETS.loc[aim, ["x", "y"]]
            hd = np.arctan2(ax_ - x, ay_ - y) + np.radians(rng.normal(0, 0.6))
        else:
            hd = rng.uniform(0, 2 * np.pi); aim = -1
        vx, vy = spd * np.sin(hd) / 1000, spd * np.cos(hd) / 1000          # km/s
        sig = {"uav_type": ((0.2, 0.5), (0.02, 0.12), [0, .1, .5, .4]), "fast_type": ((0.1, 0.4), (0.02, 0.12), [0, .2, .4, .4]),
               "decoy": ((0.3, 0.8), (0.08, 0.35), [0, .55, .4, .05]), "spoof": ((0.3, 0.8), (0.05, 0.3), [0, .7, .3, 0]),
               "civil": ((0.7, 1.0), (0.01, 0.05), [0, 0, .1, .9]), "bird": ((0.01, 0.12), (0.1, 0.4), [0, .6, .4, 0])}[typ]
        jump = spd * rng.uniform(0.95, 1.2) if typ != "spoof" else spd * rng.uniform(4, 15)
        true_tti = np.inf
        if threat:
            d_aim = np.hypot(ASSETS.loc[aim, "x"] - x, ASSETS.loc[aim, "y"] - y)
            true_tti = d_aim * 1000 / spd
        rows.append({"id": f"T{seed % 100:02d}{i:03d}", "type": typ, "x": x, "y": y, "vx": vx, "vy": vy, "alt": alt, "spd": spd,
                     "rcs": rng.uniform(*sig[0]), "rcs_var": rng.uniform(*sig[1]), "n_sensors": int(rng.choice(4, p=sig[2])),
                     "max_jump": jump, "civ_iff": int(typ == "civil" and rng.random() < 0.95),
                     "aim": aim, "threat": threat, "true_tti": true_tti,
                     "urgent": bool(threat and true_tti <= TTI_URGENT)})
    df = pd.DataFrame(rows)
    df["true_prio"] = [ASSETS.loc[a, "value"] * np.exp(-t / 300) if th else 0.0
                       for a, t, th in zip(df["aim"], df["true_tti"], df["threat"])]
    return df


def geometry(df):
    """각 항적과 가상 자산 사이 최근접점(CPA) 거리와 도달 예상시간(TTI, s)을 계산한다."""
    px, py = df["x"].to_numpy()[:, None], df["y"].to_numpy()[:, None]
    vx, vy = df["vx"].to_numpy()[:, None], df["vy"].to_numpy()[:, None]
    rx, ry = ASSETS["x"].to_numpy()[None] - px, ASSETS["y"].to_numpy()[None] - py
    v2 = vx ** 2 + vy ** 2 + 1e-12
    t = (rx * vx + ry * vy) / v2                                  # CPA 도달 시각(s)
    cpa = np.hypot(rx - vx * t, ry - vy * t)
    tti = np.where((t > 0) & (cpa < CPA_KM), t, np.inf)
    return cpa, tti

# ---------------- 위협 점수 모델(추상 모델: 거리·도달시간·자산 가치·위협 유사도) ----------------
FEATS = ["log_spd", "alt_km", "rcs", "rcs_var", "n_sensors", "civ_iff", "jump_ratio"]


def features(df):
    return pd.DataFrame({"log_spd": np.log(df["spd"]), "alt_km": df["alt"] / 1000, "rcs": df["rcs"], "rcs_var": df["rcs_var"],
                         "n_sensors": df["n_sensors"], "civ_iff": df["civ_iff"], "jump_ratio": df["max_jump"] / df["spd"]})


def train_classifier(seeds=(101, 102, 103, 104)):
    """학습용 가상 상황(시험 상황과 다른 시드)으로 '위협 유사도' 분류기를 학습한다."""
    tr = pd.concat([make_raid(s, 120, spoof=10) for s in seeds], ignore_index=True)
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, C=1.0))
    clf.fit(features(tr)[FEATS], tr["threat"].astype(int))
    return clf


def baseline_rank(df):
    """Baseline: 거리·도달시간만으로 1차 위협 점수를 매긴다(자산 가치·유형 판별 없음)."""
    _, tti = geometry(df)
    t_min = tti.min(axis=1)
    out = df[["id"]].copy()
    out["tti_s"] = t_min
    out["score"] = np.exp(-t_min / 300)
    out["urgent_pred"] = t_min <= TTI_URGENT
    return out.sort_values("score", ascending=False)


def advanced_rank(df, clf, w=None):
    """고도화: 자산 가치 x 도달시간 x 위협 유사도, 운동학 불일치(스푸핑 의심) 항적은 별도 표시한다."""
    w = w or {"tau": 300, "p_min": 0.5, "value_pow": 1.0, "jump_max": 2.0}
    _, tti = geometry(df)
    val = ASSETS["value"].to_numpy()[None] ** w["value_pow"]
    asset_score = val * np.exp(-tti / w["tau"])
    best = asset_score.argmax(axis=1)
    p = clf.predict_proba(features(df)[FEATS])[:, 1]
    spoof = (df["max_jump"] / df["spd"]).to_numpy() > w["jump_max"]
    out = df[["id"]].copy()
    out["asset"] = ASSETS["asset"].to_numpy()[best]
    out["tti_s"] = tti[np.arange(len(df)), best]
    out["p_threat"] = p
    out["spoof_flag"] = spoof
    out["score"] = np.where(spoof, 0, p * asset_score.max(axis=1))
    out["urgent_pred"] = (p >= w["p_min"]) & (out["tti_s"] <= TTI_URGENT) & ~spoof
    return out.sort_values("score", ascending=False)

CONFIG = {"w_cls": 1.0, "w_spoof": 1.0, "th_urgent": 0.37, "th_watch": 0.05, "budget_ms": 200}   # 미션 9 재보정값


def validate(df):
    ok = df["spd"].between(0.1, 400) & df["alt"].between(0, 15000) & df[["x", "y"]].abs().le(100).all(axis=1)
    return df[ok], df[~ok]


def preprocess(df):
    good, bad = validate(df)
    return good.drop_duplicates("id").reset_index(drop=True), bad


def assess(df, clf):
    _, tti = geometry(df)
    best = (ASSETS["value"].to_numpy()[None] * np.exp(-tti / 300)).argmax(axis=1)
    t = tti[np.arange(len(df)), best]
    p = clf.predict_proba(features(df)[FEATS])[:, 1]
    spoof = (df["max_jump"] / df["spd"]).to_numpy() > 2.0
    idx = np.exp(-t / 300) * p ** CONFIG["w_cls"] * (1 - CONFIG["w_spoof"] * spoof)
    lvl = np.select([idx >= CONFIG["th_urgent"], idx >= CONFIG["th_watch"]], ["긴급", "주의"], "일반")
    conf = 0.5 * p + 0.2 * df["n_sensors"].to_numpy() / 3 + 0.15 * (1 - np.clip(df["rcs_var"].to_numpy() / 0.4, 0, 1)) + 0.15 * (~spoof)
    grade = np.select([conf >= 0.7, conf >= 0.45], ["High", "Med"], "Low")
    return df.assign(asset=ASSETS["asset"].to_numpy()[best], tti_s=t, p=p, spoof_flag=spoof, index_=idx, level=lvl, conf=conf, grade=grade)


def recommend(a):
    rec = a[a["level"] != "일반"].sort_values("index_", ascending=False).reset_index(drop=True)
    rec["권고순위"] = np.arange(1, len(rec) + 1)
    rec["상태"] = "승인 대기"           # 자동 실행·자동 승인 없음
    return rec


class Queue:
    def __init__(self):
        self.items, self.log = {}, []

    def submit(self, r):
        self.items[r.id] = "PENDING"
        prev = self.log[-1]["hash"] if self.log else "GENESIS"
        body = json.dumps({"id": r.id, "rank": int(r.권고순위), "by": "AI"}, sort_keys=True)
        self.log.append({"body": body, "hash": hashlib.sha256((prev + body).encode()).hexdigest()})


def run(seed, clf):
    raw = make_raid(seed, 120, spoof=8)
    rng = np.random.default_rng(seed)
    raw.loc[rng.choice(len(raw), 4, replace=False), "spd"] = np.nan            # 손상 메시지 주입
    raw = pd.concat([raw, raw.sample(10, random_state=seed)])                 # 중복 주입
    t0 = time.perf_counter()
    df, bad = preprocess(raw)
    a = assess(df, clf)
    rec = recommend(a)
    q = Queue()
    for r in rec.itertuples():
        q.submit(r)
    return {"raw": raw, "df": df, "bad": bad, "a": a, "rec": rec, "q": q, "ms": (time.perf_counter() - t0) * 1000}


# ---------------- 1. 통합 실행 + 함수 커버리지 측정 ----------------
called = set()
mine = {"make_raid", "geometry", "features", "train_classifier", "baseline_rank", "advanced_rank", "validate", "preprocess",
        "assess", "recommend", "submit", "run"}
sys.setprofile(lambda f, e, a: called.add(f.f_code.co_name) if e == "call" and f.f_code.co_name in mine else None)
t_start = time.perf_counter()
clf = train_classifier()
results, fails = [], 0
for seed in (8, 9, 10, 11, 12):
    try:
        results.append(run(seed, clf))
    except Exception as e:
        fails += 1; print("실행 실패:", seed, e)
base_chk = baseline_rank(results[0]["df"])          # 비교 기준(Baseline) 경로도 점검에 포함
adv_chk = advanced_rank(results[0]["df"], clf)
sys.setprofile(None)
cov = len(called & mine) / len(mine)

# ---------------- 2. 성능 집계 ----------------
acc, rec_u, far = [], [], []
for R in results:
    a = R["a"]
    y = np.select([a["threat"] & (a["true_tti"] <= 300), a["threat"] & (a["true_tti"] <= 900)], ["긴급", "주의"], "일반")
    acc.append((a["level"] == y).mean())
    rec_u.append(((a["level"] != "일반") & (y == "긴급")).sum() / max((y == "긴급").sum(), 1))
    far.append((~a["threat"] & (a["level"] != "일반")).sum() / max((a["level"] != "일반").sum(), 1))
R = results[0]
print(f"[통합 실행] 상황 {len(results) + fails}개 중 성공 {len(results)}개, 평균 처리 {np.mean([r['ms'] for r in results]):.1f} ms")
print(f"  상황 1: 입력 {len(R['raw'])}건 -> 손상 격리 {len(R['bad'])}건, 중복 제거 {len(R['raw']) - len(R['bad']) - len(R['df'])}건, 평가 {len(R['df'])}건")
print(f"  우선순위 판정 정확도 {np.mean(acc):.1%}, 긴급 표적 경보 재현율 {np.mean(rec_u):.1%}, 비위협 경보 비율 {np.mean(far):.1%}")
print("\n[상황 1 권고 대응 순위 상위 5] (모두 승인 대기)")
print(R["rec"].head(5)[["권고순위", "id", "asset", "tti_s", "level", "grade", "상태"]].round({"tti_s": 0}).to_string(index=False))

# ---------------- 3. 자체 점검 ----------------
chk = {
    "E2E 파이프라인 실행 성공률 100%": fails == 0,
    "손상 메시지 전량 격리": all(r["bad"]["spd"].isna().all() and r["bad"]["id"].nunique() == 4 for r in results),
    "스푸핑 의심 항적 경보 0건": all(((r["a"]["spoof_flag"]) & (r["a"]["level"] != "일반")).sum() == 0 for r in results),
    "자동 승인 0건(전부 승인 대기)": all(set(r["q"].items.values()) <= {"PENDING"} for r in results),
    "긴급 표적 경보 재현율 90% 이상": np.mean(rec_u) >= 0.9,
    "처리 시간 예산 이내": max(r["ms"] for r in results) <= CONFIG["budget_ms"],
    "함수 커버리지 85% 이상": cov >= 0.85,
}
passed = sum(chk.values())
for k, v in chk.items():
    print(f"[{'PASS' if v else 'FAIL'}] {k}")
print(f"자체 점검 {passed}/{len(chk)} 통과, 함수 커버리지 {cov:.0%} (목표 85% 이상) -> {'달성' if cov >= 0.85 else '미달'}")

# ---------------- 4. 브리핑 상황판 ----------------
a = R["a"]
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
col = {"긴급": "red", "주의": "orange", "일반": "lightgray"}
for lv, g in a.groupby("level"):
    ax[0].scatter(g["x"], g["y"], s=14, c=col[lv], label={"긴급": "urgent", "주의": "watch", "일반": "normal"}[lv])
ax[0].scatter(ASSETS["x"], ASSETS["y"], marker="^", s=120, c="blue", label="virtual assets")
ax[0].set_aspect("equal"); ax[0].legend(fontsize=8); ax[0].set_title("Integrated picture (scenario 1)")
ax[1].axis("off")
ax[1].text(0, 0.9, f"Accuracy      {np.mean(acc):.1%}\nUrgent recall {np.mean(rec_u):.1%}\nPending recs  {len(R['rec'])}\n"
           f"Auto-approved 0\nSelf-check    {passed}/{len(chk)}\nCoverage      {cov:.0%}", family="monospace", fontsize=13, va="top")
plt.suptitle("OPS04 saturation alert system - training data only"); plt.tight_layout(); plt.show()

# ---------------- 5. 방공작전 브리핑 요약 ----------------
lv = R["a"]["level"].value_counts()
print(f"""
[포화경보 우선순위 브리핑 요약 - 교육용가상자료]
1. 상황: 가상 비행체 {len(R['df'])}건 평가, 긴급 {lv.get('긴급', 0)} / 주의 {lv.get('주의', 0)} / 일반 {lv.get('일반', 0)}
2. 성능: 판정 정확도 {np.mean(acc):.1%}, 긴급 재현율 {np.mean(rec_u):.1%}, 처리 {R['ms']:.1f} ms, 전체 소요 {time.perf_counter() - t_start:.1f} s
3. 권고: 대응 권고 {len(R['rec'])}건 모두 지휘관 승인 대기, 자동 승인 0건
4. 운용 유의: 순위는 권고이며 대응 여부는 지휘관 승인과 교전수칙·지휘계통에 따른다.
5. 한계: 추상 위협 지수와 가상 데이터 기반 결과로, 실제 운용 전에는 실제 센서 자료로 재검증해야 한다.""")


<details>
<summary><b>💡 [참고] 미션 10 예상 실행 결과 (Output)</b></summary>

```text
[통합 실행] 상황 5개 중 성공 5개, 평균 처리 31.0 ms
  상황 1: 입력 138건 -> 손상 격리 4건, 중복 제거 10건, 평가 124건
  우선순위 판정 정확도 94.5%, 긴급 표적 경보 재현율 100.0%, 비위협 경보 비율 3.3%

[상황 1 권고 대응 순위 상위 5] (모두 승인 대기)
 권고순위     id asset  tti_s level grade    상태
    1 T08115    A3   11.0    긴급  High 승인 대기
    2 T08017    A4   54.0    긴급  High 승인 대기
    3 T08094    A3   60.0    긴급  High 승인 대기
    4 T08119    A4   64.0    긴급  High 승인 대기
    5 T08015    A3   69.0    긴급  High 승인 대기
[PASS] E2E 파이프라인 실행 성공률 100%
[PASS] 손상 메시지 전량 격리
[PASS] 스푸핑 의심 항적 경보 0건
[PASS] 자동 승인 0건(전부 승인 대기)
[PASS] 긴급 표적 경보 재현율 90% 이상
[PASS] 처리 시간 예산 이내
[PASS] 함수 커버리지 85% 이상
자체 점검 7/7 통과, 함수 커버리지 100% (목표 85% 이상) -> 달성

[포화경보 우선순위 브리핑 요약 - 교육용가상자료]
1. 상황: 가상 비행체 124건 평가, 긴급 30 / 주의 30 / 일반 64
2. 성능: 판정 정확도 94.5%, 긴급 재현율 100.0%, 처리 29.1 ms, 전체 소요 1.7 s
3. 권고: 대응 권고 60건 모두 지휘관 승인 대기, 자동 승인 0건
4. 운용 유의: 순위는 권고이며 대응 여부는 지휘관 승인과 교전수칙·지휘계통에 따른다.
5. 한계: 추상 위협 지수와 가상 데이터 기반 결과로, 실제 운용 전에는 실제 센서 자료로 재검증해야 한다.
```
</details>
